In [1]:
import asyncio, json, random, re
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── proxy (your signature) ────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def build_proxy_url():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    return f"http://{PROXY_HOST}:{PROXY_PORT}", user, PROXY_PASS

def make_proxies():
    _, user, pwd = build_proxy_url()
    return {"http":  f"http://{user}:{pwd}@{PROXY_HOST}:{PROXY_PORT}",
            "https": f"http://{user}:{pwd}@{PROXY_HOST}:{PROXY_PORT}"}

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

BASE_URL = "https://it-market.com"
MAX_PAGE_CAP = 415

SUBCATEGORIES = [
    "controller-cards",
    "power-supplies",
    "processors",
    "fan-trays",
    "cables",
    "batteries",
    "adapters",
    "transceivers",
    "modems",
    "modules",
    "components-accessories",
    "other-components",
    "newcomers",
]

SORTS = ["name-asc", "name-desc", "price-desc", "price-asc", "topseller"]

# ── helpers ───────────────────────────────────────────────────────────────────
async def fetch(session, url):
    r = await session.get(url, headers=HEADERS, proxies=make_proxies(), timeout=30)
    r.raise_for_status()
    return r.text

def get_last_page(html: str) -> int:
    tree = HTMLParser(html)
    last = tree.css_first("li.page-last a")
    if last:
        p = last.attributes.get("data-page") or last.attributes.get("href", "")
        m = re.search(r'\d+', str(p))
        if m:
            return int(m.group())
    return 1

def get_total_products(html: str) -> int:
    tree = HTMLParser(html)
    # "Showing 24 out of 64354 products."
    wrapper = tree.css_first("div.js-listing-wrapper")
    if wrapper:
        aria = wrapper.attributes.get("data-aria-live-text", "")
        m = re.search(r'out of ([\d,]+)', aria)
        if m:
            return int(m.group(1).replace(",", ""))
    return 0

# ── sizing probe: fetch p=1 of each subcat, report coverage plan ──────────────
async def probe_all_subcategories():
    """
    For each subcategory, fetch page 1 with name-asc.
    Report: total products, total pages, pages needed, sorts needed, estimated coverage.
    """
    sem = asyncio.Semaphore(3)

    async def probe_one(subcat):
        async with sem:
            await asyncio.sleep(random.uniform(0.3, 0.9))
            url  = f"{BASE_URL}/en/components/{subcat}?order=name-asc&p=1"
            try:
                html      = await fetch(session, url)
                total_p   = get_total_products(html)
                last_p    = get_last_page(html)
                return subcat, total_p, last_p
            except Exception as e:
                return subcat, None, None

    async with AsyncSession(impersonate="chrome124") as session:
        results = await asyncio.gather(*[probe_one(s) for s in SUBCATEGORIES])

    print(f"\n{'Subcategory':<30} {'Products':>10} {'Pages':>8} {'Capped':>8} {'Sorts needed':>14} {'Est. coverage':>15}")
    print("─" * 90)

    plan = []
    for subcat, total_products, last_page in results:
        if total_products is None:
            print(f"{subcat:<30} {'ERROR':>10}")
            continue

        pages_available = last_page
        pages_crawlable = min(last_page, MAX_PAGE_CAP)
        products_per_sort = pages_crawlable * 24

        if pages_available <= MAX_PAGE_CAP:
            # Single sort covers everything
            sorts_needed    = ["name-asc"]
            est_unique      = total_products
            coverage_pct    = 100.0
        else:
            # Need multiple sorts; estimate unique coverage
            # Conservative: assume 60% overlap between sorts beyond first
            sorts_needed = []
            remaining    = total_products
            unique_acc   = 0
            for i, sort in enumerate(SORTS):
                if remaining <= 0:
                    break
                gain = products_per_sort if i == 0 else int(products_per_sort * 0.7)
                unique_acc += min(gain, remaining)
                remaining  -= gain
                sorts_needed.append(sort)
                if unique_acc >= total_products:
                    break
            est_unique   = min(unique_acc, total_products)
            coverage_pct = (est_unique / total_products * 100) if total_products else 0

        plan.append({
            "subcategory":   subcat,
            "total_products": total_products,
            "last_page":     pages_available,
            "sorts_needed":  sorts_needed,
            "est_unique":    est_unique,
            "coverage_pct":  coverage_pct,
        })

        flag = "✅" if coverage_pct == 100 else ("⚠️" if coverage_pct >= 70 else "❌")
        print(
            f"{subcat:<30} {total_products:>10,} {pages_available:>8,} "
            f"{pages_crawlable:>8,} {len(sorts_needed):>14} "
            f"{coverage_pct:>13.1f}% {flag}"
        )

    # Summary
    total_est   = sum(p["est_unique"] for p in plan)
    total_prods = sum(p["total_products"] for p in plan if p["total_products"])
    total_reqs  = sum(min(p["last_page"], MAX_PAGE_CAP) * len(p["sorts_needed"]) for p in plan)

    print("─" * 90)
    print(f"{'TOTAL':<30} {total_prods:>10,} {'':>8} {'':>8} {'':>14} "
          f"{total_est/total_prods*100:>13.1f}%")
    print(f"\nEstimated total PLP requests: {total_reqs:,}")
    print(f"Estimated unique products:    {total_est:,} / {total_prods:,}")
    print(f"\nFull plan (for orchestrator):")
    print(json.dumps(plan, indent=2))

    return plan

plan = asyncio.run(probe_all_subcategories())


Subcategory                      Products    Pages   Capped   Sorts needed   Est. coverage
──────────────────────────────────────────────────────────────────────────────────────────
controller-cards                    5,246      219      219              1         100.0% ✅
power-supplies                      7,910      330      330              1         100.0% ✅
processors                          6,781      283      283              1         100.0% ✅
fan-trays                           ERROR
cables                              7,273      304      304              1         100.0% ✅
batteries                             541       23       23              1         100.0% ✅
adapters                            ERROR
transceivers                       13,355      557      415              2         100.0% ✅
modems                                639       27       27              1         100.0% ✅
modules                             6,221      260      260              1         100.0%

In [3]:
# ── targeted retry for failed subcats ────────────────────────────────────────
# Run this cell first before reaching for Camoufox.
# If these 3 fail again, paste the log output and we'll know the exact cause.
import asyncio, json, logging, random, re
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# silence noisy libs
logging.getLogger("hpack").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)



FAILED_SUBCATS = ["fan-trays", "adapters", "components-accessories"]

async def retry_failed(retries: int = 3, backoff: float = 3.0):
    async with AsyncSession(impersonate="chrome124") as session:
        for subcat in FAILED_SUBCATS:
            url = f"{BASE_URL}/en/components/{subcat}?order=name-asc&p=1"
            for attempt in range(1, retries + 1):
                log.info("retry subcat=%s attempt=%d/%d", subcat, attempt, retries)
                await asyncio.sleep(backoff * attempt + random.uniform(0.5, 2.0))
                try:
                    html           = await fetch(session, url)
                    total_products = get_total_products(html)
                    last_page      = get_last_page(html)

                    if total_products == 0 and last_page == 1:
                        # parsed OK but got nothing — likely a challenge page
                        log.warning(
                            "subcat=%s got zero products — possible bot wall. "
                            "body preview:\n%s",
                            subcat, html[:600]
                        )
                        continue

                    log.info("✅ subcat=%-30s total=%6d last_page=%4d",
                             subcat, total_products, last_page)
                    break

                except Exception as e:
                    log.error("attempt %d FAILED subcat=%s %s: %s",
                              attempt, subcat, type(e).__name__, e)
                    if attempt == retries:
                        log.critical(
                            "subcat=%s exhausted %d retries — needs Camoufox fallback",
                            subcat, retries
                        )

asyncio.run(retry_failed())

03:09:41 [INFO] retry subcat=fan-trays attempt=1/3
03:09:52 [INFO] ✅ subcat=fan-trays                      total=  2104 last_page=  88
03:09:52 [INFO] retry subcat=adapters attempt=1/3
03:10:12 [INFO] ✅ subcat=adapters                       total=  6617 last_page= 276
03:10:12 [INFO] retry subcat=components-accessories attempt=1/3
03:10:33 [INFO] ✅ subcat=components-accessories         total=  7360 last_page= 307


In [4]:
import asyncio, json, logging, random, re
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
logging.getLogger("hpack").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("curl_cffi").setLevel(logging.WARNING)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def build_proxy_url():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    return f"http://{PROXY_HOST}:{PROXY_PORT}", user, PROXY_PASS

def make_proxies():
    _, user, pwd = build_proxy_url()
    url = f"http://{user}:{pwd}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

BASE_URL     = "https://it-market.com"
MAX_PAGE_CAP = 415
SORTS        = ["name-asc", "name-desc", "price-desc", "price-asc", "topseller"]

# ── top-level categories to map ───────────────────────────────────────────────
TOP_LEVEL_CATS = [
    "switches",
    "router",
    "communication",
    "servers",
    "security",
    "storage-memory",
]

# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str) -> str:
    proxies  = make_proxies()
    safe_proxy = proxies["https"].replace(PROXY_PASS, "***")
    log.debug("GET %s  via %s", url, safe_proxy)
    try:
        r = await session.get(url, headers=HEADERS, proxies=proxies, timeout=30)
        log.debug("← %d  size=%d bytes  url=%s", r.status_code, len(r.content), url)
        if r.status_code != 200:
            log.warning("non-200 %d for %s — preview: %s",
                        r.status_code, url, r.text[:300])
        r.raise_for_status()
        return r.text
    except Exception as e:
        log.error("fetch FAILED url=%s  %s: %s", url, type(e).__name__, e)
        raise

# ── parsers ───────────────────────────────────────────────────────────────────
def extract_subcats(html: str, parent_cat: str) -> list[dict]:
    """
    Pull subcategory links from the navigation flyout.
    Selector confirmed against components flyout HTML.
    Returns list of {name, url, slug} dicts.
    """
    tree  = HTMLParser(html)
    links = tree.css("a.navigation-flyout-link.is-level-0")
    log.debug("extract_subcats parent=%s  flyout links found=%d", parent_cat, len(links))

    subcats = []
    for a in links:
        href  = a.attributes.get("href", "")
        title = a.attributes.get("title", "").strip()
        name  = a.css_first("span[itemprop='name']")
        label = name.text(strip=True) if name else title

        # guard: only keep links that are children of this category
        expected_prefix = f"{BASE_URL}/en/{parent_cat}/"
        if not href.startswith(expected_prefix):
            log.debug("skipping non-child link: %s", href)
            continue

        slug = href.rstrip("/").split("/")[-1]
        subcats.append({"name": label, "url": href, "slug": slug})
        log.debug("subcat found: %-35s  %s", label, href)

    log.info("parent=%-20s  subcats discovered=%d", parent_cat, len(subcats))
    return subcats


def get_last_page(html: str) -> int:
    tree = HTMLParser(html)
    last = tree.css_first("li.page-last a")
    if not last:
        log.debug("get_last_page: no pagination → 1")
        return 1
    raw = last.attributes.get("data-page") or last.attributes.get("href", "")
    m   = re.search(r'\d+', str(raw))
    val = int(m.group()) if m else 1
    log.debug("get_last_page: %s → %d", raw, val)
    return val


def get_total_products(html: str) -> int:
    tree    = HTMLParser(html)
    wrapper = tree.css_first("div.js-listing-wrapper")
    if not wrapper:
        log.debug("get_total_products: js-listing-wrapper not found")
        return 0
    aria = wrapper.attributes.get("data-aria-live-text", "")
    log.debug("get_total_products: aria=%r", aria)
    m = re.search(r'out of ([\d,]+)', aria)
    val = int(m.group(1).replace(",", "")) if m else 0
    log.debug("get_total_products: total=%d", val)
    return val


def build_sort_plan(total_products: int, last_page: int) -> tuple[list[str], int, float]:
    """
    Given a subcat's total products and last page, determine:
      - which sorts are needed
      - estimated unique products reachable
      - coverage percentage
    Returns (sorts_needed, est_unique, coverage_pct)
    """
    pages_crawlable   = min(last_page, MAX_PAGE_CAP)
    products_per_sort = pages_crawlable * 24

    if last_page <= MAX_PAGE_CAP:
        return ["name-asc"], total_products, 100.0

    sorts_needed = []
    remaining    = total_products
    unique_acc   = 0
    for i, sort in enumerate(SORTS):
        if remaining <= 0:
            break
        gain       = products_per_sort if i == 0 else int(products_per_sort * 0.7)
        unique_acc += min(gain, remaining)
        remaining  -= gain
        sorts_needed.append(sort)
        if unique_acc >= total_products:
            break

    est_unique   = min(unique_acc, total_products)
    coverage_pct = (est_unique / total_products * 100) if total_products else 0.0
    return sorts_needed, est_unique, coverage_pct

# ── main probe ────────────────────────────────────────────────────────────────
async def map_and_probe():
    """
    For each top-level category:
      1. Fetch landing page → extract subcats from flyout + size the top-level itself
      2. Fetch p=1 of each subcat → size it
      3. Build sort plan for each node
    Produces full cat→subcat map with coverage strategy.
    """
    sem = asyncio.Semaphore(3)

    async def probe_url(url: str, retries: int = 3) -> str | None:
        """Fetch with retry+backoff, return html or None."""
        async with sem:
            for attempt in range(1, retries + 1):
                delay = (attempt - 1) * 4 + random.uniform(0.5, 2.0)
                if delay > 0:
                    log.debug("backoff %.1fs before attempt %d  url=%s", delay, attempt, url)
                    await asyncio.sleep(delay)
                try:
                    return await fetch(session, url)
                except Exception as e:
                    log.warning("attempt %d/%d failed url=%s  %s: %s",
                                attempt, retries, url, type(e).__name__, e)
            log.error("exhausted retries url=%s", url)
            return None

    full_plan   = {}   # cat → {meta, subcats: [...]}
    grand_total = 0
    grand_est   = 0
    grand_reqs  = 0

    async with AsyncSession(impersonate="chrome124") as session:

        # ── step 1: fetch all top-level landing pages concurrently ────────────
        log.info("=== STEP 1: fetching %d top-level category pages ===", len(TOP_LEVEL_CATS))
        cat_pages = {}
        tasks     = {cat: asyncio.create_task(
                        probe_url(f"{BASE_URL}/en/{cat}?order=name-asc&p=1"))
                     for cat in TOP_LEVEL_CATS}
        for cat, task in tasks.items():
            html = await task
            if html:
                cat_pages[cat] = html
                total = get_total_products(html)
                last  = get_last_page(html)
                log.info("top-level %-20s  total=%6d  last_page=%4d", cat, total, last)
            else:
                log.error("FAILED to fetch top-level cat=%s", cat)

        # ── step 2: extract subcats from each landing page ────────────────────
        log.info("=== STEP 2: extracting subcategory map ===")
        subcat_probe_tasks = []

        for cat, html in cat_pages.items():
            subcats = extract_subcats(html, cat)

            if not subcats:
                log.warning("cat=%s has NO subcats in flyout — will treat as leaf", cat)
                # treat the top-level itself as the only leaf
                total = get_total_products(html)
                last  = get_last_page(html)
                sorts, est, pct = build_sort_plan(total, last)
                full_plan[cat] = {
                    "top_level_url":  f"{BASE_URL}/en/{cat}",
                    "total_products": total,
                    "last_page":      last,
                    "subcats":        [],
                    "sorts_needed":   sorts,
                    "est_unique":     est,
                    "coverage_pct":   pct,
                }
            else:
                full_plan[cat] = {
                    "top_level_url": f"{BASE_URL}/en/{cat}",
                    "subcats":       subcats,
                }
                # queue subcat probes
                for sc in subcats:
                    url = f"{sc['url']}?order=name-asc&p=1"
                    subcat_probe_tasks.append((cat, sc, url))

        # ── step 3: probe all subcat pages concurrently ───────────────────────
        log.info("=== STEP 3: probing %d subcategories ===", len(subcat_probe_tasks))

        async def probe_subcat(cat, sc, url):
            html = await probe_url(url)
            if not html:
                log.error("FAILED subcat=%s/%s", cat, sc["slug"])
                return cat, sc, None, None
            total = get_total_products(html)
            last  = get_last_page(html)
            log.info("subcat %-15s / %-30s  total=%6d  last_page=%4d",
                     cat, sc["slug"], total, last)
            return cat, sc, total, last

        results = await asyncio.gather(*[
            probe_subcat(cat, sc, url)
            for cat, sc, url in subcat_probe_tasks
        ])

        # ── step 4: build sort plan per subcat ────────────────────────────────
        log.info("=== STEP 4: building sort plans ===")
        for cat, sc, total, last in results:
            if total is None:
                sc.update({"total_products": None, "last_page": None,
                           "sorts_needed": None, "est_unique": 0, "coverage_pct": 0})
                continue
            sorts, est, pct = build_sort_plan(total, last)
            sc.update({
                "total_products": total,
                "last_page":      last,
                "sorts_needed":   sorts,
                "est_unique":     est,
                "coverage_pct":   pct,
            })
            log.debug("plan %-15s/%-30s  sorts=%s  est=%d  cov=%.1f%%",
                      cat, sc["slug"], sorts, est, pct)

    # ── step 5: print report ──────────────────────────────────────────────────
    print(f"\n{'='*95}")
    print(f" FULL CATEGORY → SUBCATEGORY DISCOVERY & SORT STRATEGY")
    print(f"{'='*95}")

    for cat, data in full_plan.items():
        cat_total = sum(
            sc.get("total_products") or 0
            for sc in data.get("subcats", [])
        ) or data.get("total_products", 0)

        print(f"\n{'─'*95}")
        print(f"  📁 {cat.upper():<25}  ({cat_total:,} products across subcats)")
        print(f"{'─'*95}")

        if not data.get("subcats"):
            # leaf category
            sorts   = data.get("sorts_needed", [])
            est     = data.get("est_unique", 0)
            pct     = data.get("coverage_pct", 0)
            last    = data.get("last_page", 0)
            flag    = "✅" if pct == 100 else ("⚠️" if pct >= 70 else "❌")
            reqs    = min(last, MAX_PAGE_CAP) * len(sorts) if sorts else 0
            print(f"  {'(no subcats — treated as leaf)':<45} "
                  f"total={cat_total:>8,}  pages={last:>5,}  "
                  f"sorts={len(sorts)}  est={est:>8,}  {pct:>5.1f}% {flag}  reqs={reqs:,}")
            grand_total += cat_total
            grand_est   += est
            grand_reqs  += reqs
        else:
            print(f"  {'Subcategory':<35} {'Products':>10} {'Pages':>7} "
                  f"{'Sorts':>6} {'Est. unique':>12} {'Coverage':>10} {'Requests':>10}")
            print(f"  {'·'*88}")

            for sc in sorted(data["subcats"], key=lambda x: x.get("total_products") or 0, reverse=True):
                if sc.get("total_products") is None:
                    print(f"  {sc['name']:<35} {'ERROR':>10}")
                    continue
                total = sc["total_products"]
                last  = sc["last_page"]
                sorts = sc["sorts_needed"]
                est   = sc["est_unique"]
                pct   = sc["coverage_pct"]
                reqs  = min(last, MAX_PAGE_CAP) * len(sorts)
                flag  = "✅" if pct == 100 else ("⚠️" if pct >= 70 else "❌")

                print(f"  {sc['name']:<35} {total:>10,} {last:>7,} "
                      f"{len(sorts):>6} {est:>12,} {pct:>9.1f}% {flag} {reqs:>10,}")

                grand_total += total
                grand_est   += est
                grand_reqs  += reqs

    print(f"\n{'='*95}")
    print(f"  {'GRAND TOTAL':<35} {grand_total:>10,} {'':>7} "
          f"{'':>6} {grand_est:>12,} "
          f"{grand_est/grand_total*100:>9.1f}% {'':>3} {grand_reqs:>10,}")
    print(f"{'='*95}")
    print(f"\n  Est. total PLP requests : {grand_reqs:,}")
    print(f"  Est. unique products    : {grand_est:,} / {grand_total:,}")

    return full_plan

full_plan = asyncio.run(map_and_probe())

03:19:35 [INFO] === STEP 1: fetching 6 top-level category pages ===
03:19:35 [DEBUG] backoff 1.4s before attempt 1  url=https://it-market.com/en/switches?order=name-asc&p=1
03:19:35 [DEBUG] backoff 1.4s before attempt 1  url=https://it-market.com/en/router?order=name-asc&p=1
03:19:35 [DEBUG] backoff 0.8s before attempt 1  url=https://it-market.com/en/communication?order=name-asc&p=1
03:19:35 [DEBUG] GET https://it-market.com/en/communication?order=name-asc&p=1  via http://brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.0942335402785585:***@brd.superproxy.io:22225
03:19:36 [DEBUG] GET https://it-market.com/en/router?order=name-asc&p=1  via http://brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.2377498866091303:***@brd.superproxy.io:22225
03:19:36 [DEBUG] GET https://it-market.com/en/switches?order=name-asc&p=1  via http://brd-customer-hl_14d32bce-zone-datacenter_proxy1-session-0.3507460466392077:***@brd.superproxy.io:22225
03:19:50 [DEBUG] ← 200  size=555766 bytes  u


 FULL CATEGORY → SUBCATEGORY DISCOVERY & SORT STRATEGY

───────────────────────────────────────────────────────────────────────────────────────────────
  📁 SWITCHES                   (24,271 products across subcats)
───────────────────────────────────────────────────────────────────────────────────────────────
  Subcategory                           Products   Pages  Sorts  Est. unique   Coverage   Requests
  ························································································
  Gigabit                                 11,014     459      2       11,014     100.0% ✅        830
  Other Switches                           7,362     307      1        7,362     100.0% ✅        307
  10/100 MBit/s                            3,485     146      1        3,485     100.0% ✅        146
  Managed                                  1,396      59      1        1,396     100.0% ✅         59
  Unmanaged                                  610      26      1          610     100.0% ✅    

In [14]:
import asyncio, json, logging, random, re
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def make_proxies():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    url   = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

PRICE_RE = re.compile(r'[\d,]+\.?\d*')

def parse_price(text: str):
    if not text or "request" in text.lower():
        return "price_on_request"
    # handle european format: €2,380.00 or €107.10
    clean = text.replace("€", "").replace("*", "").strip()
    # remove thousands comma: 2,380.00 → 2380.00
    clean = re.sub(r',(?=\d{3})', '', clean)
    m = re.search(r'[\d]+\.[\d]+|[\d]+', clean)
    return float(m.group()) if m else None

def parse_itmarket_page(html: str, nav_page_url: str, input_url: str) -> list[dict]:
    tree     = HTMLParser(html)
    cards    = tree.css("div.product-box")
    products = []

    for card in cards:
        # ── identifiers ───────────────────────────────────────────────────────
        a           = card.css_first("a.product-name")
        product_url = a.attributes.get("href") if a else None
        # href is already absolute from debug log
        if product_url and not product_url.startswith("http"):
            product_url = "https://it-market.com" + product_url

        name_input   = card.css_first("input[name='product-name']")
        id_input     = card.css_first("input[name='product-id']")
        product_code = name_input.attributes.get("value", "").strip() if name_input else None
        product_id   = id_input.attributes.get("value", "").strip()   if id_input   else None

        # ── brand + breadcrumb from url path ──────────────────────────────────
        # pattern: /en/{cat}/{subcat}/{subcat2}/{brand}/{slug}
        brand = breadcrumb = None
        if product_url:
            parts = product_url.rstrip("/").split("/")
            # parts[0]='' parts[1]='' parts[2]='it-market.com' ... parts[3]='en'
            # or for absolute: ['https:', '', 'it-market.com', 'en', cat, subcat, subcat2, brand, slug]
            try:
                en_idx = parts.index("en")
                rel    = parts[en_idx+1:]   # [cat, subcat, subcat2, brand, slug]
                brand  = rel[-2].capitalize() if len(rel) >= 2 else None
                breadcrumb = {
                    "category":     rel[0] if len(rel) > 0 else None,
                    "subcategory":  rel[1] if len(rel) > 1 else None,
                    "subcategory2": rel[2] if len(rel) > 2 else None,
                }
            except (ValueError, IndexError):
                pass

        # ── description ───────────────────────────────────────────────────────
        desc_node   = card.css_first("div.product-description")
        description = desc_node.text(strip=True) if desc_node else None

        # ── variants ──────────────────────────────────────────────────────────
        # selector confirmed: div.product-detail-configurator-option
        # condition text in:  div.maxia-variants-list-text  (first line before "with delivery time")
        # price in:           div.product-variant-price
        # stock in:           div.product-variant-stock
        variants = []
        prices   = {}

        for opt in card.css("div.product-detail-configurator-option"):
            # condition: first word of maxia-variants-list-text
            text_node = opt.css_first("div.maxia-variants-list-text")
            cond_raw  = text_node.text(strip=True) if text_node else ""
            # e.g. "Refurbishedwith delivery time" or "Refurbishedin stock"
            condition = re.split(r'(?=[A-Z])', cond_raw)[0].strip().lower() if cond_raw else None

            price_node = opt.css_first("div.product-variant-price")
            price_text = price_node.text(strip=True) if price_node else ""
            price      = parse_price(price_text)

            stock_node = opt.css_first("div.product-variant-stock")
            stock_text = stock_node.text(strip=True) if stock_node else ""
            stock_m    = re.search(r'\d+', stock_text)
            stock      = int(stock_m.group()) if stock_m else 0

            # variant_id not on the option div in new template — use product-id
            # (no data-product-id attribute visible in debug output)
            variants.append({
                "condition": condition,
                "price":     price,
                "stock":     stock,
            })
            if condition and price and price != "price_on_request":
                prices[condition] = price

        products.append({
            "product_url":  product_url,
            "product_name": product_code,
            "brand":        brand,
            "model":        product_code,
            "product_code": product_code,
            "product_id":   product_id,
            "ean_upc":      None,
            "breadcrumb":   breadcrumb,
            "description":  description,
            "prices":       prices,
            "variants":     variants,
            "input_url":    input_url,
            "nav_page_url": nav_page_url,
        })

    return products


async def itmarket_single_page_sample():
    url = "https://it-market.com/en/components/transceivers?order=name-asc&p=1"

    async with AsyncSession(impersonate="chrome124") as session:
        r = await session.get(
            url,
            headers=HEADERS,
            proxies=make_proxies(),
            timeout=30,
        )
        log.info("status=%d  size=%d bytes", r.status_code, len(r.content))
        r.raise_for_status()
        html = r.text

    products = parse_itmarket_page(
        html,
        nav_page_url=url,
        input_url="https://it-market.com/en/components/transceivers",
    )

    log.info("parsed %d products", len(products))
    print(json.dumps(products, indent=2, ensure_ascii=False))
    output_path = "itmarket_sample_page1.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(products, f, indent=2, ensure_ascii=False)
    log.info("saved %d products to %s", len(products), output_path)

asyncio.run(itmarket_single_page_sample())

04:30:25 [INFO] status=200  size=654305 bytes
04:30:25 [INFO] parsed 24 products
04:30:25 [INFO] saved 24 products to itmarket_sample_page1.json


[
  {
    "product_url": "https://it-market.com/en/components/transceivers/other-transceivers/dell/001-ssc-9791",
    "product_name": "001-SSC-9791",
    "brand": "Dell",
    "model": "001-SSC-9791",
    "product_code": "001-SSC-9791",
    "product_id": "4f4f5b5ff1146f5474768f58796a6dd5",
    "ean_upc": null,
    "breadcrumb": {
      "category": "components",
      "subcategory": "transceivers",
      "subcategory2": "other-transceivers"
    },
    "description": "DELL SONICWALL 100BASE-T SFP COPPER MODULE",
    "prices": {},
    "variants": [
      {
        "condition": "",
        "price": "price_on_request",
        "stock": 0
      },
      {
        "condition": "",
        "price": "price_on_request",
        "stock": 0
      }
    ],
    "input_url": "https://it-market.com/en/components/transceivers",
    "nav_page_url": "https://it-market.com/en/components/transceivers?order=name-asc&p=1"
  },
  {
    "product_url": "https://it-market.com/en/components/transceivers/other-tran

In [3]:
import asyncio, json, logging, random, re, time
import nest_asyncio
nest_asyncio.apply()

from curl_cffi.requests import AsyncSession
from selectolax.parser import HTMLParser

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── proxy ─────────────────────────────────────────────────────────────────────
PROXY_HOST = "brd.superproxy.io"
PROXY_PORT  = 22225
PROXY_USER  = "brd-customer-hl_14d32bce-zone-datacenter_proxy1"
PROXY_PASS  = "ymg5cg3a1z33"

def make_proxies():
    token = f"{random.random():.16f}"
    user  = f"{PROXY_USER}-session-{token}"
    url   = f"http://{user}:{PROXY_PASS}@{PROXY_HOST}:{PROXY_PORT}"
    return {"http": url, "https": url}

HEADERS = {
    "accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "referer":         "https://www.google.com/",
}

BASE_URL     = "https://it-market.com"
CATEGORY_URL = "https://it-market.com/en/switches"
MAX_PAGES    = 5
OUTPUT_FILE  = "itmarket_switches_p1_p5.json"

# ── price parser (EU format) ──────────────────────────────────────────────────
def parse_price(text: str):
    """
    Handles:
      €65.45*       →  65.45
      €1,041.25*    →  1041.25   (English thousands separator)
      €1.041,25*    →  1041.25   (German: dot=thousands, comma=decimal)
      price on request → "price_on_request"
    """
    t = text.strip().replace("*", "").replace("\xa0", "").replace("€", "").strip()
    if not t or "request" in t.lower():
        return "price_on_request"

    # German format: 1.041,25
    if re.search(r'\d\.\d{3},\d', t):
        t = t.replace(".", "").replace(",", ".")
    # English thousands: 1,041.25
    elif re.search(r'\d,\d{3}\.?\d*', t):
        t = t.replace(",", "")
    # Simple German decimal: 65,45
    elif "," in t and "." not in t:
        t = t.replace(",", ".")

    m = re.search(r'\d+(?:\.\d+)?', t)
    return float(m.group()) if m else None


# ── variant label parser ──────────────────────────────────────────────────────
def parse_variant_label(opt) -> tuple[str, str]:
    text_div = opt.css_first("div.maxia-variants-list-text")
    if not text_div:
        label = opt.css_first("label")
        raw   = label.attributes.get("title", "unknown") if label else "unknown"
        return raw, "unknown"

    # Availability from span class (do this first so we can strip it from the text)
    avail_span = text_div.css_first("span")
    if avail_span and "inStock" in avail_span.attributes.get("class", ""):
        availability = "in stock"
    elif avail_span and "withDelivery" in avail_span.attributes.get("class", ""):
        availability = "with delivery time"
    else:
        availability = "unknown"

    # Condition = full text minus the span's text
    full_text   = text_div.text(strip=True)          # "Refurbishedin stock"
    span_text   = avail_span.text(strip=True) if avail_span else ""
    condition   = full_text.replace(span_text, "").strip()  # "Refurbished"

    return condition, availability

# ── PLP parser ────────────────────────────────────────────────────────────────
def parse_itmarket_page(html: str, nav_page_url: str, input_url: str) -> list[dict]:
    tree     = HTMLParser(html)
    cards    = tree.css("div.product-box")
    products = []

    log.info("  cards found: %d", len(cards))

    for card in cards:

        # ── identifiers ───────────────────────────────────────────────────────
        a           = card.css_first("a.product-name")
        product_url = a.attributes.get("href") if a else None
        if product_url and not product_url.startswith("http"):
            product_url = BASE_URL + product_url

        name_input   = card.css_first("input[name='product-name']")
        id_input     = card.css_first("input[name='product-id']")
        product_code = name_input.attributes.get("value", "").strip() if name_input else None
        product_id   = id_input.attributes.get("value", "").strip()   if id_input   else None

        # ── brand + breadcrumb from URL path ──────────────────────────────────
        brand = None
        breadcrumb = {}
        if product_url:
            parts = product_url.rstrip("/").split("/")
            try:
                en_idx = parts.index("en")
                rel    = parts[en_idx + 1:]   # [cat, subcat?, subcat2?, brand, slug]
                brand  = rel[-2].capitalize() if len(rel) >= 2 else None
                breadcrumb = {
                    "category":    rel[0] if len(rel) > 0 else None,
                    "subcategory": rel[1] if len(rel) > 1 else None,
                    "subcategory2":rel[2] if len(rel) > 2 else None,
                }
            except (ValueError, IndexError):
                pass

        # ── description ───────────────────────────────────────────────────────
        desc_node   = card.css_first("div.product-description")
        description = desc_node.text(strip=True) if desc_node else None

        # ── variants: condition + availability + price + stock ─────────────────
        variants = []
        prices   = {}   # flat summary: "Refurbished (in_stock)": 65.45

        for opt in card.css("div.product-detail-configurator-option"):
            variant_id           = opt.attributes.get("data-product-id", "") or None
            condition, avail     = parse_variant_label(opt)

            price_node  = opt.css_first("p.product-price")
            price_text  = price_node.text(strip=True) if price_node else ""
            price       = parse_price(price_text)

            stock_node  = opt.css_first("div.product-variant-stock")
            stock_text  = stock_node.text(strip=True) if stock_node else ""
            stock_m     = re.search(r'\d+', stock_text)
            stock       = int(stock_m.group()) if stock_m else 0

            key = f"{condition} ({avail})"
            variants.append({
                "variant_product_id": variant_id,
                "condition":          condition,
                "availability":       avail,
                "price":              price,
                "stock":              stock,
            })
            prices[key] = price

        products.append({
            "product_url":  product_url,
            "product_name": product_code,
            "brand":        brand,
            "model":        product_code,
            "product_code": product_code,
            "product_id":   product_id,
            "ean_upc":      None,
            "breadcrumb":   breadcrumb,
            "description":  description,
            "prices":       prices,
            "variants":     variants,
            "input_url":    input_url,
            "nav_page_url": nav_page_url,
        })

    return products


# ── fetch ─────────────────────────────────────────────────────────────────────
async def fetch(session: AsyncSession, url: str) -> str:
    t0 = time.perf_counter()
    r  = await session.get(url, headers=HEADERS, proxies=make_proxies(), timeout=30)
    log.info("  GET %s  →  %d  %.2fs  %d bytes",
             url, r.status_code, time.perf_counter() - t0, len(r.content))
    r.raise_for_status()
    return r.text


# ── main ──────────────────────────────────────────────────────────────────────
async def scrape_switches():
    all_products = []

    async with AsyncSession(impersonate="chrome124") as session:
        for page in range(1, MAX_PAGES + 1):
            url = f"{CATEGORY_URL}?min-price=0&p={page}"
            log.info("── page %d/%d ─────────────────────────────", page, MAX_PAGES)
            try:
                html  = await fetch(session, url)
                prods = parse_itmarket_page(html, nav_page_url=url, input_url=CATEGORY_URL)
                all_products.extend(prods)
                log.info("  parsed %d products  (running total: %d)", len(prods), len(all_products))
            except Exception as e:
                log.error("  page %d FAILED: %s", page, e)

            if page < MAX_PAGES:
                await asyncio.sleep(random.uniform(1.0, 2.5))

    # ── save ──────────────────────────────────────────────────────────────────
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    log.info("══ done: %d products saved to %s ══", len(all_products), OUTPUT_FILE)

    # ── quick sanity print ────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  Total products : {len(all_products)}")
    print(f"  Output file    : {OUTPUT_FILE}")
    print(f"{'='*60}")
    if all_products:
        p = all_products[0]
        print(f"\nSample — {p['product_name']}")
        for v in p["variants"]:
            print(f"  [{v['condition']:<14} | {v['availability']:<20}]  "
                  f"€{v['price']}  stock={v['stock']}")

asyncio.run(scrape_switches())

08:27:07 [INFO] ── page 1/5 ─────────────────────────────
08:27:12 [INFO]   GET https://it-market.com/en/switches?min-price=0&p=1  →  200  4.77s  623678 bytes
08:27:12 [INFO]   cards found: 24
08:27:12 [INFO]   parsed 24 products  (running total: 24)
08:27:13 [INFO] ── page 2/5 ─────────────────────────────
08:27:15 [INFO]   GET https://it-market.com/en/switches?min-price=0&p=2  →  200  1.43s  626308 bytes
08:27:15 [INFO]   cards found: 24
08:27:15 [INFO]   parsed 24 products  (running total: 48)
08:27:16 [INFO] ── page 3/5 ─────────────────────────────
08:27:18 [INFO]   GET https://it-market.com/en/switches?min-price=0&p=3  →  200  1.95s  620920 bytes
08:27:18 [INFO]   cards found: 24
08:27:18 [INFO]   parsed 24 products  (running total: 72)
08:27:20 [INFO] ── page 4/5 ─────────────────────────────
08:27:21 [INFO]   GET https://it-market.com/en/switches?min-price=0&p=4  →  200  1.69s  616440 bytes
08:27:21 [INFO]   cards found: 24
08:27:21 [INFO]   parsed 24 products  (running total: 


  Total products : 120
  Output file    : itmarket_switches_p1_p5.json

Sample — 2530-24-PoE+
  [Refurbished    | in stock            ]  €71.4  stock=30
  [Refurbished    | with delivery time  ]  €136.85  stock=10
  [New            | with delivery time  ]  €1071.0  stock=5
